# Jour 3 — TP : un CSV, un Parquet, et on analyse

*Formation Python pour l'analyse de données — sujet*

Deux fichiers dans `donnees/` :

- **`devis.csv`** : les devis du jour 2 (client, produit, date d'édition, montant, statut),
  au format d'un export SAS / Excel français ;
- **`clients_devis.parquet`** : le **référentiel clients** (âge, segment, région, canal d'arrivée).
  D'habitude on irait le chercher dans la base de données ; ici, quelqu'un l'a extrait **une fois**
  en Parquet, et on travaille sur le fichier.

Objectif : relier les deux, et chercher **qui accepte nos devis**.

Équivalences SAS : `PROC IMPORT` → `pd.read_csv`, `LIBNAME` + table → `pd.read_parquet`,
`MERGE` / `PROC SQL JOIN` → `merge`, `PROC MEANS` / `PROC TABULATE` → `groupby` / `pivot_table`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DONNEES = Path("donnees")

## 1. Lire le CSV  ★☆☆

Lisez `donnees/devis.csv` dans `devis`. Ouvrez-le d'abord dans l'explorateur de fichiers
(ou `!head -3 donnees/devis.csv`) pour trouver : le séparateur, la décimale, l'encodage
(export Windows), le format des dates.

Vérifiez avec `devis.dtypes` : `montant` doit être un nombre et `date_edition` une date.

In [ ]:
# devis = pd.read_csv(DONNEES / "devis.csv", ...)   # à compléter

## 2. Nettoyer (rappel du jour 2)  ★☆☆

En une cellule : supprimez les doublons, harmonisez `statut` (`ACCEPTE` / `REFUSE`),
écartez les montants négatifs, et ajoutez une colonne booléenne `accepte`.

In [ ]:
# À vous

## 3. Lire le Parquet  ★☆☆

1. Lisez `donnees/clients_devis.parquet` dans `clients` avec `pd.read_parquet`.
2. Regardez `clients.dtypes` : qu'avez-vous eu à préciser, comparé au CSV ? Pourquoi ?
3. Relisez le fichier en ne gardant que `num_client` et `canal` (`columns=`), puis seulement les
   clients du `Grand Est` (`filters=[("region", "==", "Grand Est")]`). C'est l'équivalent d'un
   `SELECT ... WHERE` dans la base.

In [ ]:
# À vous

## 4. Relier devis et clients  ★★☆

1. Joignez `devis` et `clients` sur `num_client` dans `complet` (`merge`, `how="left"`).
   Ajoutez `validate="m:1"` (plusieurs devis pour un client, un client par ligne du référentiel)
   et `indicator=True`.
2. Combien de devis n'ont **pas** trouvé leur client (`_merge == "left_only"`) ? Quels clients ?
   Ce sont des anomalies à signaler.
3. Combien de clients du référentiel n'ont **aucun** devis ?

In [ ]:
# À vous

## 5. Qui accepte nos devis ?  ★★☆

1. Taux d'acceptation (%) par **canal**, par **segment** et par **région** (trois `groupby`).
   Lequel des trois fait vraiment une différence ?
2. Tableau croisé **canal × produit** du taux d'acceptation (`pivot_table`, `aggfunc="mean"`).

In [ ]:
# À vous

## 6. L'âge du client  ★★☆

1. Calculez l'âge du client à la date du devis (en années, `(date_edition - date_naissance).dt.days / 365.25`).
2. Découpez en tranches `18-30`, `30-45`, `45-60`, `60+` (`pd.cut`).
3. Taux d'acceptation et montant moyen par tranche, puis un boxplot du montant par tranche,
   coloré par statut (seaborn).

In [ ]:
# À vous

## 7. Sauvegarder en Parquet  ★★☆

1. Enregistrez `complet` en Parquet (`to_parquet("devis_enrichis.parquet", index=False)`)
   **et** en CSV (format français).
2. Comparez la taille des deux fichiers (`Path(...).stat().st_size`).
3. Relisez les deux et comparez les types : lequel a gardé les dates et les catégories ?
4. Mesurez le temps de relecture avec `%timeit pd.read_parquet(...)` et `%timeit pd.read_csv(...)`.

In [ ]:
# À vous

## 8. Un Parquet par produit (partitionner)  ★★★

1. Écrivez `complet` découpé par produit : `complet.to_parquet("devis_par_produit", partition_cols=["produit"])`.
   Regardez le dossier créé (`sorted(Path("devis_par_produit").rglob("*.parquet"))`).
2. Relisez **seulement** la santé : `pd.read_parquet("devis_par_produit", filters=[("produit", "==", "SANTE")])`.
   Combien de lignes ? Les autres sous-dossiers n'ont même pas été ouverts.
3. Pourquoi est-ce utile quand les données font des millions de lignes ?

In [ ]:
# À vous